### Library Import

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import time
import random
from datetime import datetime
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

In [7]:
import re
import time
import random
from datetime import datetime
from urllib.parse import urljoin, urlparse

import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from playwright.sync_api import sync_playwright


class InvestingNewsCrawler:
    def __init__(
        self,
        base_url: str = "https://www.investing.com",
        news_url: str = "https://www.investing.com/news",
        headless: bool = False,
        sleep_min: float = 1.0,
        sleep_max: float = 2.5,
        timeout: int = 60000,
    ):
        self.base_url = base_url
        self.news_url = news_url
        self.headless = headless
        self.sleep_min = sleep_min
        self.sleep_max = sleep_max
        self.timeout = timeout

    @staticmethod
    def _clean_text(text: str | None) -> str | None:
        if text is None:
            return None

        text = re.sub(r"\s+", " ", text).strip()
        return text if text else None

    @staticmethod
    def _is_article_url(url: str) -> bool:
        path = urlparse(url).path

        if not path.startswith("/news/"):
            return False

        exclude_keywords = [
            "/news/latest-news",
            "/news/most-popular-news",
            "/news/stock-market-news",
            "/news/forex-news",
            "/news/commodities-news",
            "/news/economy",
            "/news/cryptocurrency-news",
        ]

        if path in exclude_keywords:
            return False

        # Investing.com 기사 URL은 보통 끝에 숫자 id가 붙음
        return bool(re.search(r"-\d{5,}$", path))

    def _sleep(self):
        time.sleep(random.uniform(self.sleep_min, self.sleep_max))

    def _new_context(self, browser):
        return browser.new_context(
            viewport={"width": 1440, "height": 1000},
            locale="en-US",
            timezone_id="America/New_York",
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/126.0.0.0 Safari/537.36"
            ),
            extra_http_headers={
                "accept-language": "en-US,en;q=0.9",
                "referer": "https://www.google.com/",
            },
        )

    def collect_article_links(self, page, max_links: int = 30) -> list[str]:
        page.goto(self.news_url, wait_until="domcontentloaded", timeout=self.timeout)
        page.wait_for_timeout(3000)

        # 쿠키 팝업이 있으면 닫기
        for text in ["Accept All", "I Accept", "Agree", "Accept"]:
            try:
                page.get_by_text(text, exact=False).click(timeout=1500)
                page.wait_for_timeout(1000)
                break
            except Exception:
                pass

        # 스크롤을 내려서 lazy loaded article link 확보
        for _ in range(5):
            page.mouse.wheel(0, 1200)
            page.wait_for_timeout(1000)

        html = page.content()
        soup = BeautifulSoup(html, "lxml")

        links = []
        seen = set()

        for a in soup.select("a[href]"):
            href = a.get("href")
            if not href:
                continue

            full_url = urljoin(self.base_url, href)
            path = urlparse(full_url).path

            if not self._is_article_url(full_url):
                continue

            if path in seen:
                continue

            seen.add(path)
            links.append(full_url)

            if len(links) >= max_links:
                break

        return links

    def parse_article(self, page, url: str) -> dict:
        page.goto(url, wait_until="domcontentloaded", timeout=self.timeout)
        page.wait_for_timeout(2500)

        html = page.content()
        soup = BeautifulSoup(html, "lxml")

        headline = self._extract_headline(soup)
        author = self._extract_author(soup)
        published_at_raw = self._extract_published_at(soup)
        body = self._extract_body(soup, headline=headline)

        title_tag = soup.find("title")
        page_title = self._clean_text(title_tag.get_text(" ", strip=True)) if title_tag else None

        return {
            "source": "investing.com",
            "category": "breaking_news",
            "headline": headline,
            "author": author,
            "published_at_raw": published_at_raw,
            "body": body,
            "url": url,
            "page_title": page_title,
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "error": None,
        }

    def _extract_headline(self, soup: BeautifulSoup) -> str | None:
        h1 = soup.find("h1")
        if h1:
            return self._clean_text(h1.get_text(" ", strip=True))

        og_title = soup.find("meta", property="og:title")
        if og_title and og_title.get("content"):
            return self._clean_text(og_title.get("content"))

        return None

    def _extract_author(self, soup: BeautifulSoup) -> str | None:
        candidates = []

        for selector in [
            '[class*="author"]',
            '[data_lake-test*="author"]',
            'span[class*="Author"]',
            'a[href*="/members/"]',
        ]:
            for tag in soup.select(selector):
                txt = self._clean_text(tag.get_text(" ", strip=True))
                if txt:
                    candidates.append(txt)

        for txt in candidates:
            if len(txt) <= 80 and not txt.lower().startswith("by "):
                return txt
            if txt.lower().startswith("by "):
                return txt[3:].strip()

        text = soup.get_text("\n", strip=True)
        m = re.search(r"\bBy\s+([A-Za-z가-힣 .,'-]{2,80})", text)
        if m:
            return self._clean_text(m.group(1))

        return None

    def _extract_published_at(self, soup: BeautifulSoup) -> str | None:
        text = soup.get_text("\n", strip=True)
        lines = [self._clean_text(x) for x in text.split("\n")]
        lines = [x for x in lines if x]

        for line in lines:
            if line.startswith("Published "):
                return line.replace("Published ", "").strip()

        patterns = [
            r"Published\s+([A-Za-z]{3,9}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2})",
            r"Published\s+(.+?\d{1,2}:\d{2}.+?)",
            r"Updated\s+(.+?\d{1,2}:\d{2}.+?)",
        ]

        for p in patterns:
            m = re.search(p, text)
            if m:
                return self._clean_text(m.group(1))

        return None

    def _extract_body(self, soup: BeautifulSoup, headline: str | None = None) -> str | None:
        paragraphs = []

        selectors = [
            "article p",
            "main p",
            'div[class*="article"] p',
            'div[class*="Article"] p',
            'div[class*="content"] p',
        ]

        for selector in selectors:
            for p in soup.select(selector):
                txt = self._clean_text(p.get_text(" ", strip=True))
                if txt:
                    paragraphs.append(txt)

            if len(paragraphs) >= 3:
                break

        if not paragraphs:
            for p in soup.find_all("p"):
                txt = self._clean_text(p.get_text(" ", strip=True))
                if txt:
                    paragraphs.append(txt)

        bad_patterns = [
            "Sign In",
            "Free Sign Up",
            "Advertisement",
            "View all comments",
            "Comment Guidelines",
            "Risk Disclosure",
            "Trading in financial instruments",
            "Fusion Media",
            "Get real-time updates",
            "Trade With A Regulated Broker",
            "In this article:",
            "Related Articles",
            "Related Quotes",
            "Editor's Picks",
            "Most Popular News",
            "Download the App",
            "All rights reserved",
        ]

        cleaned = []
        seen = set()

        for txt in paragraphs:
            if headline and txt == headline:
                continue

            if any(bad.lower() in txt.lower() for bad in bad_patterns):
                continue

            if len(txt) < 40:
                continue

            if txt in seen:
                continue

            seen.add(txt)
            cleaned.append(txt)

        if not cleaned:
            return None

        return "\n".join(cleaned)

    def crawl(self, max_links: int = 30) -> pd.DataFrame:
        rows = []

        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=self.headless,
                args=[
                    "--disable-blink-features=AutomationControlled",
                    "--disable-dev-shm-usage",
                    "--no-sandbox",
                ],
            )

            context = self._new_context(browser)
            page = context.new_page()

            article_links = self.collect_article_links(page, max_links=max_links)

            print(f"collected article links: {len(article_links)}")

            for url in tqdm(article_links, desc="collecting investing.com articles"):
                try:
                    row = self.parse_article(page, url)
                    rows.append(row)

                except Exception as e:
                    rows.append(
                        {
                            "source": "investing.com",
                            "category": "breaking_news",
                            "headline": None,
                            "author": None,
                            "published_at_raw": None,
                            "body": None,
                            "url": url,
                            "page_title": None,
                            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "error": str(e),
                        }
                    )

                self._sleep()

            context.close()
            browser.close()

        df = pd.DataFrame(rows)

        columns = [
            "source",
            "category",
            "headline",
            "author",
            "published_at_raw",
            "body",
            "url",
            "page_title",
            "collected_at",
            "error",
        ]

        for col in columns:
            if col not in df.columns:
                df[col] = None

        df = df[columns]
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

        return df

In [11]:
if __name__ == "__main__":
    crawler = InvestingNewsCrawler(
        headless=False,   # 처음에는 False로 두는 게 좋음. 브라우저 뜨는지 확인 가능.
    )
    news_data = crawler.crawl(max_links = 10)

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.